# Linear-Probe Baseline — Legal-BERT PT-BR (base)

Frozen-encoder linear-probe baseline for `dominguesm/legal-bert-base-cased-ptbr` (`src.config.MODEL_REGISTRY["legalbert"]`) on the.

## Section 1 — Setup & Data Loading

Imports, the pinned `legalbert` entry from `src.config.MODEL_REGISTRY`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path("../..").resolve()))
from src.capture import capture_full_attention_subset, capture_logits_and_cls
from src.config import MODEL_REGISTRY
from src.data import build_datasets, load_label_map, load_splits
from src.evaluate import evaluate_predictions, full_test_report
from src.finetune_multilayer import MultiLayerClsClassifier
from src.linear_probe import (
    build_frozen_encoder_and_tokenizer,
    extract_embeddings,
    history_to_dicts,
    train_linear_probe,
    training_args_payload,
)
from src.plotting import plot_confusion_matrix
from src.runtime import get_device, set_all_seeds

set_all_seeds()

In [ ]:
model_cfg = MODEL_REGISTRY["legalbert"]

df = load_splits()
LABELS = load_label_map()["labels"]

device = get_device()
print(f"device: {device}")
print(f"split sizes: {df['split'].value_counts().to_dict()}")
print(f"labels ({len(LABELS)}): {LABELS}")

## Section 2 — Step 1: Load the Frozen Encoder

`build_frozen_encoder_and_tokenizer` loads the pretrained encoder with `output_hidden_states=True` and freezes every parameter.

In [ ]:
model, tokenizer = build_frozen_encoder_and_tokenizer(model_cfg, device)
model

## Section 3 — Step 2: The Forward Pass and Tuple Extraction

Wrapped in `torch.no_grad()` since the encoder is frozen and this is inference, not training.

In [ ]:
sample_texts = df["Sumario"].iloc[:4].tolist()

encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=model_cfg.max_length,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = model(**encoded)

hidden_states = outputs.hidden_states
print(f"num hidden_states tensors: {len(hidden_states)}")
print(f"shape of each tensor: {hidden_states[0].shape}")

## Section 4 — Step 3: Isolate the Last N Layers and the `[CLS]` Token

Lower layers skew toward raw grammar, so only the last 4 layers are used (the standard approach).

In [ ]:
LAST_N_LAYERS = 4

last_layers = hidden_states[-LAST_N_LAYERS:]
stacked = torch.stack(last_layers, dim=0)
cls_stacked = stacked[:, :, 0, :]
cls_stacked.shape

## Section 5 — Step 4: Apply the Pooling Strategy

In [ ]:
def pool_concat(cls_stacked: torch.Tensor) -> torch.Tensor:
    """Concatenate the [CLS] representations of the last N layers end-to-end.

    cls_stacked: (n_layers, batch_size, hidden_size) -> (batch_size, n_layers * hidden_size)
    """
    n_layers, batch_size, hidden_size = cls_stacked.shape
    return cls_stacked.permute(1, 0, 2).reshape(batch_size, n_layers * hidden_size)

pooled_concat = pool_concat(cls_stacked)
pooled_concat.shape

In [ ]:
def embed_sumarios(texts: list[str], last_n_layers: int = LAST_N_LAYERS) -> torch.Tensor:
    """Steps 2-4 end to end: tokenize -> forward pass -> last-N-layer [CLS] concat pooling.

    Returns a (len(texts), last_n_layers * hidden_size) tensor.
    """
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=model_cfg.max_length,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoded)

    stacked = torch.stack(outputs.hidden_states[-last_n_layers:], dim=0)
    cls_stacked = stacked[:, :, 0, :]
    return pool_concat(cls_stacked)

## Section 6 — Step 5: Extract Cached Embeddings & Train the Linear Probe Head

`src.data.build_datasets` tokenizes `Sumario` and attaches the numeric `label_id` for every row.

In [ ]:
datasets = build_datasets(df, tokenizer, max_length=model_cfg.max_length)
{split: len(ds) for split, ds in datasets.items()}

In [ ]:
extract_batch_size = model_cfg.batch_size(device.type)

train_embeddings = extract_embeddings(model, datasets["train"], device, batch_size=extract_batch_size)
val_embeddings = extract_embeddings(model, datasets["val"], device, batch_size=extract_batch_size)

print(f"train_embeddings: {tuple(train_embeddings.shape)}")
print(f"val_embeddings  : {tuple(val_embeddings.shape)}")

In [ ]:
PROBE_NUM_EPOCHS = 20
PROBE_LEARNING_RATE = 1e-3
PROBE_WEIGHT_DECAY = 0.01
PROBE_BATCH_SIZE = 64

head, probe_history = train_linear_probe(
    train_embeddings, datasets["train"].label_ids,
    val_embeddings, datasets["val"].label_ids,
    num_labels=len(LABELS), labels=LABELS, device=device,
    num_epochs=PROBE_NUM_EPOCHS, learning_rate=PROBE_LEARNING_RATE,
    weight_decay=PROBE_WEIGHT_DECAY, batch_size=PROBE_BATCH_SIZE,
)

## Section 7 — Full Test-Split Results, XAI Artifacts & Metrics Report

The trained head is reattached to the frozen encoder via `MultiLayerClsClassifier`.

In [ ]:
test_df = df[df["split"] == "test"].reset_index(drop=True)

combined_model = MultiLayerClsClassifier(
    model, model.config.hidden_size, len(LABELS), dropout=0.0, n_layers=LAST_N_LAYERS,
).to(device)
combined_model.classifier = head
combined_model.eval()

results_dir = model_cfg.results_dir("linear_probe")
batch_size = model_cfg.batch_size(device.type)

full_split_artifacts = capture_logits_and_cls(combined_model, datasets["test"], device, batch_size=batch_size)
for name, array in full_split_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

In [ ]:
N_ATTENTION_SUBSET = 50

attention_artifacts = capture_full_attention_subset(
    combined_model, datasets["test"], device, n_samples=N_ATTENTION_SUBSET
)
for name, array in attention_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

attention_subset_ids = test_df["ImportJurisprudenciaId"].iloc[:N_ATTENTION_SUBSET].tolist()
with open(results_dir / "attention_subset_ids.json", "w", encoding="utf-8") as f:
    json.dump(attention_subset_ids, f, indent=2)

In [ ]:
test_label_ids = test_df["label_id"].tolist()
test_pred_ids = full_split_artifacts["logits"].argmax(axis=-1).tolist()

test_metrics = evaluate_predictions(test_label_ids, test_pred_ids, LABELS)
test_report = full_test_report(test_label_ids, test_pred_ids, LABELS)

test_metrics

In [ ]:
checkpoint_dir = results_dir / "checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
torch.save(head.state_dict(), checkpoint_dir / "linear_probe_head_state_dict.pt")
tokenizer.save_pretrained(checkpoint_dir)

results_df = test_df[["ImportJurisprudenciaId", "Decisao"]].copy()
results_df["predicted_label"] = [LABELS[i] for i in test_pred_ids]
results_df["confidence"] = full_split_artifacts["probs"].max(axis=-1).tolist()
results_df.to_csv(results_dir / "results.csv", index=False)

metrics_payload = {
    "epoch_history": history_to_dicts(probe_history),
    "summary": test_metrics,
    "classification_report": test_report["classification_report"],
    "confusion_matrix": test_report["confusion_matrix"],
    "confusion_matrix_labels": test_report["confusion_matrix_labels"],
}
with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

plot_confusion_matrix(
    test_report["confusion_matrix"], LABELS, results_dir / "confusion_matrix.png",
    title=f"{model_cfg.display_name} — {results_dir.name}",
)

args_payload = training_args_payload(
    model_cfg, device,
    num_epochs=PROBE_NUM_EPOCHS, learning_rate=PROBE_LEARNING_RATE,
    weight_decay=PROBE_WEIGHT_DECAY, batch_size=PROBE_BATCH_SIZE,
    epochs_run=len(probe_history),
)
with open(results_dir / "training_args.json", "w", encoding="utf-8") as f:
    json.dump(args_payload, f, indent=2)

print(f"saved {len(results_df)} predictions + metrics report to {results_dir}")